# PE2 (ESCC): protein-adjusted phosphosite analysis

`PE2.xlsx`, sheets `2.a protein raw information` and `3.a phospho raw information`: per-sample
relative abundances for 24 normal (`ESCC-xxxN`) and 94 tumor (`ESCC-xxxT`) samples. All 24 normals
have a tumor from the same patient; 70 tumors have no normal.

Tumor vs normal, three analyses: protein, phospho, and phospho adjusted for protein (Method A,
with Method B as a sensitivity check).

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import phosphokit as pk

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
IN_FILE = Path("../../data/raw/PE2/PE2.xlsx")
DATASET = "PE2"
OUT_DIR = Path(f"../../results/{DATASET}")
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

S = pk.Settings(normalization="quantile_common", missing_method="none", min_per_group=3)
GROUP_COLORS = {"N": "#55A868", "T": "#C44E52"}


def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight")
    plt.show()

## 1. Load

In [ ]:
prot_lin, prot_annot = pk.io.read_pe2_sheet(IN_FILE, "2.a protein raw information")
ph_lin, ph_annot = pk.io.read_pe2_sheet(IN_FILE, "3.a phospho raw information")

# proteins keyed by UniProt accession, phosphosites by accession-residue-position
prot_annot = prot_annot.set_index("Protein accession")
ph_annot = ph_annot.set_index("ID")
prot_lin.index, ph_lin.index = prot_annot.index, ph_annot.index

samples = pk.io.pe2_sample_table(ph_lin.columns)
paired_cases = samples.groupby("Case")["Group"].nunique().loc[lambda s: s == 2].index
print(f"proteins {prot_lin.shape[0]}, phosphosites {ph_lin.shape[0]}, samples {ph_lin.shape[1]}")
print(samples["Group"].value_counts().to_dict(), f"| patients with N and T: {len(paired_cases)}")

## 2. Normalization

The bias table is the argument for `quantile_common`: most features should not move between groups,
so the median log2FC of the common features should stay near 0.

In [ ]:
prot_raw, ph_raw = np.log2(prot_lin), np.log2(ph_lin)
bias = pd.concat({"protein": pk.normalization_bias(prot_raw, samples["Group"], reference="N"),
                  "phospho": pk.normalization_bias(ph_raw, samples["Group"], reference="N")}, axis=1)
print(bias.round(3))

prot_log = pk.normalize(prot_raw, S.normalization, S.norm_common_frac)
ph_log = pk.normalize(ph_raw, S.normalization, S.norm_common_frac)
prot_log, _ = pk.impute(prot_log, S.missing_method, S.impute_min_valid_frac, S)
ph_log, _ = pk.impute(ph_log, S.missing_method, S.impute_min_valid_frac, S)

fig, axes = plt.subplots(2, 2, figsize=(15, 6))
pk.sample_boxplot(prot_raw, axes[0, 0], samples["Group"], GROUP_COLORS, "Protein, raw")
pk.sample_boxplot(prot_log, axes[0, 1], samples["Group"], GROUP_COLORS, f"Protein, {S.normalization}")
pk.sample_boxplot(ph_raw, axes[1, 0], samples["Group"], GROUP_COLORS, "Phospho, raw")
pk.sample_boxplot(ph_log, axes[1, 1], samples["Group"], GROUP_COLORS, f"Phospho, {S.normalization}")
savefig("01_boxplots")

## 3. Protein adjustment input

Sites are linked to their protein by UniProt accession, which is more specific than the gene name.

In [ ]:
site_to_protein = ph_annot["Protein accession"]
ph_adj = pk.adjusted_matrix(ph_log, prot_log, site_to_protein)
protein_by_site = prot_log.loc[site_to_protein[ph_adj.index].values].set_axis(ph_adj.index)
print(f"adjustable sites: {len(ph_adj)} / {len(ph_annot)} ({len(ph_adj) / len(ph_annot):.1%})")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
site_r = pk.phospho_protein_correlation(ph_log.loc[ph_adj.index], protein_by_site, ax)
savefig("02_phospho_protein_correlation")

## 4. Tumor vs normal

limma with `duplicateCorrelation(block = patient)`: the 24 paired patients contribute one normal and
one tumor sample, so the paired information is used without dropping the 70 tumor-only patients.

In [ ]:
res, info = {}, {}
res["protein"], info["protein"] = pk.two_group_test(prot_log, samples, ["T"], ["N"], S)
res["phospho"], info["phospho"] = pk.two_group_test(ph_log, samples, ["T"], ["N"], S)
res["adjA"], info["adjA"] = pk.two_group_test(ph_adj, samples, ["T"], ["N"], S)
res["adjB"] = pk.method_b(res["phospho"], res["protein"], site_to_protein)

overview = pd.DataFrame([{"analysis": k, "tested": len(r),
                          "Up": (pk.direction(r, S) == "Up").sum(),
                          "Down": (pk.direction(r, S) == "Down").sum(),
                          "rho": info.get(k, {}).get("rho"), "prior_d0": info.get(k, {}).get("d0")}
                         for k, r in res.items()])
overview

## 5. QC

`Ratio` in the source file is `2^(mean log2 T - mean log2 N)` of the same values, so the logFC here
should track it closely; the differences come from normalization and patient blocking.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (k, annot) in zip(axes, [("protein", prot_annot), ("phospho", ph_annot)]):
    paper = np.log2(annot.loc[res[k].index, "Ratio"])
    ax.scatter(paper, res[k]["logFC"], s=3, alpha=0.3)
    lim = np.nanpercentile(np.abs(paper), 99.9)
    ax.plot([-lim, lim], [-lim, lim], color="red", lw=0.8)
    ax.set_xlabel("log2 Ratio (source file)")
    ax.set_ylabel("limma logFC")
    ax.set_title(f"{k}: r = {paper.corr(res[k]['logFC']):.3f}")
savefig("03_crosscheck_source_ratio")

pk.pvalue_histograms(res)
savefig("04_pvalue_histograms")

In [ ]:
site_label = (ph_annot["Gene name"].fillna("").astype(str) + " "
              + ph_annot["Amino acid"].astype(str) + ph_annot["Postion"].astype(str)).to_dict()
gene_label = prot_annot["Gene name"].to_dict()

fig, axes = plt.subplots(1, 4, figsize=(20, 4.8))
for ax, (k, title, lab) in zip(axes, [("protein", "Protein", gene_label),
                                      ("phospho", "Phosphosite (unadjusted)", site_label),
                                      ("adjA", "Adjusted, Method A", site_label),
                                      ("adjB", "Adjusted, Method B", site_label)]):
    pk.volcano(res[k], ax, title, labels=lab, settings=S)
savefig("05_volcano")

pk.save_volcanoes({k: res[k] for k in ["protein", "phospho", "adjA"]},
                  FIG_DIR / "volcano_individual",
                  labels={"protein": gene_label, "phospho": site_label, "adjA": site_label}, settings=S)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
pk.method_comparison(res["adjA"], res["adjB"], axes[0], axes[1], site_correlation=site_r)
savefig("06_methodA_vs_B")

## 6. Tables

Both the unadjusted and the adjusted DEP list are exported. They answer different questions, and
sites whose protein was not quantified exist only in the unadjusted list.

In [ ]:
def pick(result, prefix, cols=("logFC", "P", "adj.P")):
    """Selected columns of a result table, prefixed, plus the Up/Down call."""
    out = result[list(cols)].add_prefix(f"{prefix}_")
    out[f"{prefix}_direction"] = pk.direction(result, S)
    return out


annot = ph_annot[["Protein accession", "Gene name", "Postion", "Amino acid", "Protein description"]] \
    .rename(columns={"Postion": "Position"})
annot["Source_log2Ratio"] = np.log2(ph_annot["Ratio"])

table = annot.loc[res["adjA"].index].join(pick(res["phospho"], "Phospho"))
table = table.join(pick(res["protein"], "Protein", cols=("logFC", "P")), on="Protein accession")
table = table.join(pick(res["adjA"], "Adjusted"))
table["Adjusted_MethodB_direction"] = pk.direction(res["adjB"], S).reindex(table.index)
table["Category"] = pk.category_table(res["protein"], res["phospho"], res["adjA"], site_to_protein, S)["category"]
table = table.sort_values("Adjusted_P")

phospho_dep = annot.join(pick(res["phospho"], "Phospho"), how="inner")
phospho_dep["Protein_quantified"] = phospho_dep.index.isin(ph_adj.index)
phospho_dep = phospho_dep[phospho_dep["Phospho_direction"] != ""].sort_values("Phospho_P")
adjusted_dep = table[table["Adjusted_direction"] != ""]

print(table["Category"].value_counts())
print(f"phospho DEP {len(phospho_dep)} | adjusted DEP {len(adjusted_dep)}")

out_file = OUT_DIR / f"{DATASET}_phospho_results.xlsx"
with pd.ExcelWriter(out_file) as xw:
    table.to_excel(xw, sheet_name="Adjusted_site_table")
    phospho_dep.to_excel(xw, sheet_name="Phospho_DEP")
    adjusted_dep.to_excel(xw, sheet_name="Adjusted_DEP")
    ph_adj.round(4).to_excel(xw, sheet_name="Adjusted_matrix_log2")
    prot_annot[["Gene name"]].join(pick(res["protein"], "Protein"), how="inner") \
        .sort_values("Protein_P").to_excel(xw, sheet_name="Protein")
    overview.to_excel(xw, sheet_name="Overview", index=False)
print("saved:", out_file)